# Ocena zapisanych checkpointów U-Neta

Notebook ocenia wybrany checkpoint na zapisanym podziale walidacyjnym LGG MRI. Jedno przejście modelu dostarcza predykcji do pięciu metryk obejmujących przekroje i pełne wolumeny pacjentów. Wyniki po największej składowej 3D raportujemy razem z wynikami surowej predykcji.

Wybór pliku checkpointu określa oceniany przebieg i epokę. Pole `preprocessing` w checkpointach ze standaryzacją uruchamia odpowiednie przetwarzanie danych. Starsze checkpointy korzystają z dzielenia intensywności przez 255.

Do odtworzenia oceny wykonaj komórki od góry w kernelu **Python (inzynierka)**. Eksport pobiera nazwę przebiegu ze ścieżki checkpointu. Porównania zapisanych eksperymentów są w notebookach 09 i 10. Ocenę zbioru testowego zaplanowano po uzgodnionym zamknięciu eksperymentów i zamrożeniu protokołu.

## 1. Biblioteki, dane i checkpoint

Łączymy metadane przekrojów z zapisanym podziałem pacjentów i wybieramy `val`. Definiujemy odczyt TIFF-ów oraz funkcję obliczającą statystyki kanałów na wszystkich przekrojach jednego pacjenta. Średnie i odchylenia liczymy w `float64`.

W komórce z `checkpoint_path` wskazujemy plik do oceny. Wagi wczytujemy na CPU do U-Neta i ustawiamy `eval()`. Wydruk podaje nazwę checkpointu, seed, epokę i regułę preprocessingu.

In [14]:
from pathlib import Path
import sys

import numpy as np
from scipy import ndimage

import torch
import pandas as pd
import tifffile
from torch.utils.data import Dataset, DataLoader

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models import UNet

In [15]:
pairs_df = pd.read_csv(project_root / "data" / "metadata" / "pairs.csv")
splits_df = pd.read_csv(project_root / "data" / "splits" / "splits.csv")
pairs_with_split = pairs_df.merge(
    splits_df, on="patient_id", how="left", validate="many_to_one"
)

val_df = pairs_with_split.loc[pairs_with_split["split"] == "val"].copy()

In [16]:
def compute_patient_stats(patient_df, project_root):
    patient_df = patient_df.sort_values("slice_id")

    volume = np.stack([
        tifffile.imread(project_root / path)
        for path in patient_df["image_path"]
    ]).astype(np.float32) / 255

    m = volume.mean(axis=(0, 1, 2), dtype=np.float64)
    s = volume.std(axis=(0, 1, 2), dtype=np.float64)

    return m, s

In [17]:
def preprocess_pair(image, mask):
    image_tensor = torch.from_numpy(image).permute(2, 0, 1).float() / 255
    mask_tensor = torch.from_numpy(mask > 0).float().unsqueeze(0)

    return (image_tensor, mask_tensor)

class MRIDataset(Dataset):
    def __init__(self, pairs_df, project_root, transform=None):
        self.pairs_df = pairs_df
        self.project_root = project_root
        self.transform = transform

    def __len__(self):
        return self.pairs_df.shape[0]

    def __getitem__(self, index):
        row = self.pairs_df.iloc[index]
        image_path = self.project_root / row["image_path"]
        mask_path = self.project_root / row["mask_path"]
        image, mask = preprocess_pair(tifffile.imread(image_path), tifffile.imread(mask_path))

        if self.transform is not None:
            return self.transform(image, mask)
        return image, mask

In [59]:
checkpoint_path = (
    project_root / "outputs" / "checkpoints" / "unet_sampling_zscore_seed44_last.pt"
)
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
model = UNet()
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Checkpoint: {checkpoint_path.name}, seed: {checkpoint['seed']}, epoka: {checkpoint['epoch']}")
print("Preprocessing:", checkpoint.get("preprocessing", "scale_255"))

Checkpoint: unet_sampling_zscore_seed44_last.pt, seed: 44, epoka: 40
Preprocessing: patient_channel_zscore


## 2. Predykcje i definicje Dice

Loader zachowuje kolejność przekrojów pacjenta. Model zwraca logity, a próg `sigmoid(logits) > 0.5` wyznacza predykcję binarną. Łączymy paczki po osi przekrojów w wolumen `(N, 1, H, W)`.

Dice liczymy jako dwukrotność przecięcia podzieloną przez sumę rozmiarów obu masek. Dwie puste maski otrzymują 1, a para z dokładnie jedną pustą maską otrzymuje 0.

Raportujemy średnią po wszystkich przekrojach, po przekrojach z adnotacją we wzorcu i po pustych maskach wzorcowych. Dice pustych masek odpowiada udziałowi pustych przekrojów z całkowicie pustą predykcją. Dice 3D wyznaczamy z sum dla całego wolumenu pacjenta, a następnie uśredniamy wyniki pacjentów.

In [60]:
@torch.inference_mode()
def predict_patient(model, patient_loader):
    prediction_batches = []
    mask_batches = []

    for images, masks in patient_loader:
        probabilities = torch.sigmoid(model(images))
        prediction_batches.append(probabilities > 0.5)
        mask_batches.append(masks > 0)

    return torch.cat(prediction_batches, dim=0), torch.cat(mask_batches, dim=0)

In [61]:
def dice_per_sample(predictions, targets):
    targets_logic = targets > 0

    targets_sum = targets_logic.sum(dim=(1, 2, 3))
    predictions_sum = predictions.sum(dim=(1, 2, 3))
    intersection = (targets_logic & predictions).sum(dim=(1, 2, 3))

    denominator = targets_sum + predictions_sum

    return torch.where(
        denominator == 0,
        1.0,
        2 * intersection / denominator.clamp_min(1)
    )


def dice3d(preds, masks):
    intersection = (preds & masks).sum().item()
    preds_count = preds.sum().item()
    masks_count = masks.sum().item()

    denominator = preds_count + masks_count

    dice = 1.0 if denominator == 0 else 2 * intersection / denominator

    return (intersection, preds_count, masks_count, dice)

## 3. Największy spójny obszar 3D

Etykietujemy obszary w całym wolumenie predykcji przy sąsiedztwie 6 i zachowujemy największy z nich. Etykieta 0 oznacza tło. Funkcja obsługuje także pustą predykcję.

Filtrowanie może usuwać odizolowane fałszywe oznaczenia oraz części poprawnej predykcji. Dlatego obie wersje porównujemy z tą samą maską referencyjną i zachowujemy oba wyniki.

In [62]:
def keep_largest_component(preds):
    labeled, n_components = ndimage.label(preds.squeeze(1).numpy())
    if n_components == 0:
        return preds

    sizes = np.bincount(labeled.ravel())
    largest_label = sizes[1:].argmax() + 1
    return torch.from_numpy(labeled == largest_label).unsqueeze(1)

## 4. Preprocessing pacjenta i ocena walidacyjna

Dla każdego pacjenta sortujemy przekroje po `slice_id`. Przy `patient_channel_zscore` najpierw obliczamy jego trzy średnie i trzy odchylenia z obrazów podzielonych przez 255, uwzględniając wszystkie piksele i przekroje. Następnie stosujemy te statystyki do kolejnych przekrojów. `partial` wiąże statystyki z transformacją wywoływaną przez `MRIDataset`. Obliczenia standaryzacji mają precyzję `float64`, a wejście modelu ma typ `float32`.

Funkcja oceny przyjmuje wybór standaryzacji odczytany z metadanych checkpointu. Dla starszych zapisów stosuje preprocessing z dzieleniem przez 255. Walidacja obejmuje 16 pacjentów i 615 przekrojów, w tym 236 z adnotacją.

W metrykach 2D każdy przekrój ma tę samą wagę. W metrykach 3D każdy pacjent ma tę samą wagę w średniej. Augmentacje treningowe są oddzielnym etapem przetwarzania.

In [63]:
from functools import partial

def standardize_pair(image, mask, m, s):
    mean = torch.as_tensor(m, dtype=torch.float64).reshape(3, 1, 1)
    std = torch.as_tensor(s, dtype=torch.float64).reshape(3, 1, 1)

    return ((image - mean) / std).float(), mask

In [64]:
def evaluate_patients_3d(model, split_df, project_root, use_zscore=False):
    patient_results = []
    dice_batches = []
    annotation_batches = []

    for patient_id, patient_rows in split_df.groupby("patient_id"):
        patient_df = patient_rows.sort_values("slice_id")
        
        transform = None
        if use_zscore:
            m, s = compute_patient_stats(patient_df, project_root)
            transform = partial(standardize_pair, m=m, s=s)

        dataset = MRIDataset(
            patient_df, project_root, transform=transform
        )
        
        loader = DataLoader(dataset, batch_size=4, shuffle=False)
        preds, masks = predict_patient(model, loader)

        dice_batches.append(dice_per_sample(preds, masks))
        annotation_batches.append(masks.flatten(1).any(dim=1))

        filtered_preds = keep_largest_component(preds)
        patient_results.append({
            "patient_id": patient_id,
            "dice_3d": dice3d(preds, masks)[-1],
            "dice_3d_filtered": dice3d(filtered_preds, masks)[-1],
        })

    return pd.DataFrame(patient_results), torch.cat(dice_batches), torch.cat(annotation_batches)

In [65]:
patient_results, all_dice, all_annotations = evaluate_patients_3d(
    model,
    val_df,
    project_root,
    use_zscore=checkpoint.get("preprocessing") == "patient_channel_zscore",
)
print(
    f"Pacjenci: {len(patient_results)}, przekroje: {all_dice.numel()}, "
    f"z adnotacją: {all_annotations.sum().item()}"
)

metrics = pd.Series({
    "Dice po wszystkich przekrojach": all_dice.mean().item(),
    "Dice przekrojów z adnotacją": all_dice[all_annotations].mean().item(),
    "Dice przekrojów z pustą maską": all_dice[~all_annotations].mean().item(),
    "Dice 3D po pacjentach": patient_results["dice_3d"].mean(),
    "Dice 3D po pacjentach po filtrowaniu": patient_results["dice_3d_filtered"].mean(),
}, name="Walidacja")
metrics.to_frame().round(4)

Pacjenci: 16, przekroje: 615, z adnotacją: 236


,Walidacja
Dice po wszystkich przekrojach,0.7777
Dice przekrojów z adnotacją,0.7045
Dice przekrojów z pustą maską,0.8232
Dice 3D po pacjentach,0.6916
Dice 3D po pacjentach po filtrowaniu,0.7119


## 5. Wyniki poszczególnych pacjentów

Tabela pokazuje Dice 3D surowej predykcji oraz wynik po zachowaniu największego spójnego obszaru dla każdego pacjenta. Pozwala sprawdzić, u których osób postprocessing pomaga, a u których pogarsza wynik.

Średnia walidacyjna opisuje bieżący checkpoint. Przy porównaniu przebiegów zachowujemy ten sam podział pacjentów, próg, definicje metryk i regułę postprocessingu. Informację o kryterium wyboru epoki zapisujemy w opisie serii eksperymentów.

In [66]:
patient_results.round(4)

,patient_id,dice_3d,dice_3d_filtered
0,TCGA_CS_5396_20010302,0.3699,0.3697
1,TCGA_CS_6665_20010817,0.5654,0.5900
2,TCGA_DU_5849_19950405,0.8205,0.9034
3,TCGA_DU_6407_19860514,0.8334,0.9241
4,TCGA_DU_7013_19860523,0.0776,0.0776
5,TCGA_DU_7304_19930325,0.7924,0.7920
6,TCGA_DU_8164_19970111,0.7338,0.7454
7,TCGA_DU_A5TP_19970614,0.8087,0.8367
8,TCGA_DU_A5TU_19980312,0.7830,0.7994
9,TCGA_FG_6691_20020405,0.8491,0.9264


## 6. Zapis wyników

Eksportujemy pięć metryk zbiorczych i wyniki poszczególnych pacjentów do `outputs/`. Nazwa bazowa pochodzi z `checkpoint_path.stem`. Usuwamy końcowe `_best`, a sufiks `_last` zachowujemy, dzięki czemu oceny wybranego checkpointu i ostatniej epoki mają osobne pliki.

Eksport wykonujemy po wczytaniu checkpointu i ponownym obliczeniu jego metryk. Przy ponownej ocenie tego samego pliku aktualizujemy odpowiadające mu CSV.

In [67]:
result_name = checkpoint_path.stem.removesuffix("_best")

metrics.to_csv(
    project_root / "outputs" / f"{result_name}_val_metrics.csv"
)
patient_results.to_csv(
    project_root / "outputs" / f"{result_name}_val_patients.csv",
    index=False,
)